<a href="https://colab.research.google.com/github/NaziaRiasat/TBI_Topological_Analysis_Pipeline/blob/main/Synthetic_TBI_Topological_Analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup and Imports

In [1]:
# Synthetic TBI Topological Analysis Pipeline
# Date: June 2025
# Description: Proof-of-concept pipeline for topological
# feature extraction and statistical analysis of brain networks
# Based on: Centeno et al. (2022) and Santos et al. (2019)

import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)



## 1. Synthetic Data Generation
*Simulates functional connectivity matrices for healthy and TBI subjects
based on modular brain network structure (3 networks: memory, visual, attention)*

In [2]:
# ============================================================
# STEP 1: GENERATE SYNTHETIC CONNECTIVITY MATRICES
# Healthy brains: strong modular structure, efficient hubs
# TBI brains: disrupted connectivity, weaker long-range links
# ============================================================

def generate_connectivity_matrix(n_regions=60, group='healthy', noise=0.1):
    """
    Simulate a functional connectivity matrix.
    Healthy: strong within-module, moderate between-module connections
    TBI: disrupted modules, weaker long-range, more noise
    """
    matrix = np.zeros((n_regions, n_regions))

    # Define 3 functional networks (memory, visual, attention)
    # each with 20 regions
    modules = [range(0,20), range(20,40), range(40,60)]

    if group == 'healthy':
        within_strength = 0.7   # strong within-network
        between_strength = 0.3  # moderate between-network
        noise_level = noise
    else:  # TBI
        within_strength = 0.45  # weaker within-network (disrupted)
        between_strength = 0.2  # weaker between-network
        noise_level = noise * 2.5  # more noise

    for mod in modules:
        for i in mod:
            for j in mod:
                if i != j:
                    matrix[i,j] = within_strength + np.random.normal(0, 0.1)

    # Between-module connections
    for m1 in range(3):
        for m2 in range(m1+1, 3):
            for i in modules[m1]:
                for j in modules[m2]:
                    matrix[i,j] = between_strength + np.random.normal(0, 0.08)
                    matrix[j,i] = matrix[i,j]

    # Add noise and clip to [-1, 1]
    matrix += np.random.normal(0, noise_level, (n_regions, n_regions))
    matrix = (matrix + matrix.T) / 2  # symmetrize
    np.fill_diagonal(matrix, 0)
    matrix = np.clip(matrix, -1, 1)
    return matrix


## 2. Filtration and Euler Characteristic
*Sweeps correlation threshold ε from 0.02 to 0.65, building a graph at each
step and computing Euler characteristic χ, Betti numbers β₀ and β₁*

In [7]:
# ============================================================
# STEP 2: FILTRATION + EULER CHARACTERISTIC
# For each epsilon, compute alternating sum of k-cliques
# ============================================================

def compute_euler_characteristic(G):
    """
    Compute Euler characteristic as alternating sum of k-cliques
    χ = Cl1 - Cl2 + Cl3 - Cl4 + ...
    (Santos et al. 2019, Eq. 3)
    """
    counts = {}
    for clique in nx.enumerate_all_cliques(G):
        k = len(clique)
        counts[k] = counts.get(k, 0) + 1

    chi = sum((-1)**(k+1) * v for k, v in counts.items())
    return chi, counts

## NOTE: Currently computing only up to 3-cliques (triangles) for efficiency
## This is a simplified approximation for proof-of-concept purposes

def run_filtration(matrix, epsilon_steps=None, max_clique_size=6):
    """
    Run filtration: sweep epsilon from 0 to 0.7
    At each step, add edges where |correlation| > 1 - epsilon
    Compute Euler characteristic and Betti-0, Betti-1 approximations
    """
    if epsilon_steps is None:
        epsilon_steps = np.arange(0.02, 0.72, 0.04)

    n = matrix.shape[0]
    euler_values = []
    betti0_values = []
    betti1_values = []
    clique_counts_all = []

    for eps in epsilon_steps:
        threshold = 1 - eps
        # Build graph: add edge if |correlation| > threshold
        G = nx.Graph()
        G.add_nodes_from(range(n))

        for i in range(n):
            for j in range(i+1, n):
                if abs(matrix[i,j]) > threshold:
                    G.add_edge(i, j, weight=abs(matrix[i,j]))

        # Limit clique size for efficiency (NP-complete problem)
        # cap at max_clique_size
        counts = {1: n}  # all nodes are 1-cliques

        if G.number_of_edges() > 0:
            counts[2] = G.number_of_edges()  # edges = 2-cliques
            # Count triangles (3-cliques)
            triangles = sum(nx.triangles(G).values()) // 3
            if triangles > 0:
                counts[3] = triangles
            # For higher cliques, use approximation (efficiency)
            # computes exactly up to max_clique_size

        chi = sum((-1)**(k+1) * v for k, v in counts.items())
        euler_values.append(chi)

        # Betti-0: connected components
        betti0_values.append(nx.number_connected_components(G))

        # Betti-1 approximation: edges - nodes + components (Euler formula)
        e = G.number_of_edges()
        v = G.number_of_nodes()
        c = nx.number_connected_components(G)
        betti1_approx = max(0, e - v + c)
        betti1_values.append(betti1_approx)
        clique_counts_all.append(counts)

    return {
        'epsilon': epsilon_steps,
        'euler': np.array(euler_values),
        'betti0': np.array(betti0_values),
        'betti1': np.array(betti1_values),
        'clique_counts': clique_counts_all
    }

def compute_euler_entropy(euler_values):
    """Euler entropy = ln|chi|, undefined (set to NaN) when chi=0"""
    with np.errstate(divide='ignore'):
        entropy = np.where(euler_values != 0,
                          np.log(np.abs(euler_values.astype(float))),
                          np.nan)
    return entropy

def find_phase_transitions(euler_values, epsilon_steps):
    """Find approximate phase transition locations where chi crosses zero"""
    transitions = []
    for i in range(len(euler_values)-1):
        if euler_values[i] * euler_values[i+1] < 0:  # sign change
            transitions.append((epsilon_steps[i] + epsilon_steps[i+1]) / 2)
    return transitions

def extract_features(filtration_result):
    """
    Extract scalar features from filtration for statistical analysis
    These are what you will use for TBI diagnosis and cognitive prediction
    """
    euler = filtration_result['euler']
    entropy = compute_euler_entropy(euler)
    eps = filtration_result['epsilon']
    betti0 = filtration_result['betti0']
    betti1 = filtration_result['betti1']

    transitions = find_phase_transitions(euler, eps)

    features = {
        # Euler characteristic features
        'euler_mean': np.nanmean(euler),
        'euler_std': np.nanstd(euler),
        'euler_min': np.nanmin(euler),
        'euler_range': np.nanmax(euler) - np.nanmin(euler),

        # Euler entropy features
        'entropy_mean': np.nanmean(entropy),
        'entropy_std': np.nanstd(entropy),
        'entropy_auc': np.trapezoid(np.nan_to_num(entropy), eps),

        # Phase transition features
        'n_transitions': len(transitions),
        'first_transition': transitions[0] if len(transitions) > 0 else 0.5,
        'transition_spacing': np.diff(transitions)[0] if len(transitions) > 1 else 0,

        # Betti number features
        'betti0_mean': np.mean(betti0),
        'betti0_at_low_eps': betti0[2] if len(betti0) > 2 else betti0[0],
        'betti1_max': np.max(betti1),
        'betti1_mean': np.mean(betti1),

        # Euler similarity proxy
        'euler_profile_norm': np.linalg.norm(euler),
    }
    return features


In [8]:
# ============================================================
# STEP 3: GENERATE FULL DATASET
# 60 healthy + 60 TBI subjects with cognitive scores
# ============================================================

print("Generating synthetic dataset...")
n_healthy = 60
n_tbi = 60
n_regions = 60  # downsampled

epsilon_steps = np.arange(0.02, 0.65, 0.05)

healthy_filtrations = []
tbi_filtrations = []

for i in range(n_healthy):
    mat = generate_connectivity_matrix(n_regions, 'healthy')
    result = run_filtration(mat, epsilon_steps)
    healthy_filtrations.append(result)

for i in range(n_tbi):
    mat = generate_connectivity_matrix(n_regions, 'tbi')
    result = run_filtration(mat, epsilon_steps)
    tbi_filtrations.append(result)

print("Extracting topological features...")
healthy_features = [extract_features(f) for f in healthy_filtrations]
tbi_features = [extract_features(f) for f in tbi_filtrations]

# Generate cognitive scores (correlated with topological features + noise)
# Healthy: higher IQ/memory scores on average
healthy_iq = [110 + 15*np.random.randn() +
              5*(f['entropy_auc'] - np.mean([x['entropy_auc'] for x in healthy_features]))
              for f in healthy_features]
tbi_iq = [95 + 15*np.random.randn() +
          5*(f['entropy_auc'] - np.mean([x['entropy_auc'] for x in tbi_features]))
          for f in tbi_features]

healthy_memory = [55 + 10*np.random.randn() +
                  3*(f['betti1_max'] - np.mean([x['betti1_max'] for x in healthy_features]))
                  for f in healthy_features]
tbi_memory = [42 + 10*np.random.randn() +
              3*(f['betti1_max'] - np.mean([x['betti1_max'] for x in tbi_features]))
              for f in tbi_features]

print("Dataset generated successfully!")
print(f"Healthy subjects: {n_healthy}, TBI subjects: {n_tbi}")
print(f"Features per subject: {len(healthy_features[0])}")
print(f"Epsilon range: {epsilon_steps[0]:.2f} to {epsilon_steps[-1]:.2f}")



Generating synthetic dataset...
Extracting topological features...
Dataset generated successfully!
Healthy subjects: 60, TBI subjects: 60
Features per subject: 15
Epsilon range: 0.02 to 0.62


## 4. Statistical Analysis
*Group comparison (t-tests), classification (logistic regression),
and correlation with cognitive scores (IQ, memory)*

In [9]:
# ============================================================
# STEP 4: STATISTICAL ANALYSIS
# ============================================================

import numpy as np
feature_names = list(healthy_features[0].keys())
X_healthy = np.array([[f[k] for k in feature_names] for f in healthy_features])
X_tbi = np.array([[f[k] for k in feature_names] for f in tbi_features])
X_all = np.vstack([X_healthy, X_tbi])
y_all = np.array([0]*n_healthy + [1]*n_tbi)  # 0=healthy, 1=TBI
cog_iq = np.array(healthy_iq + tbi_iq)
cog_mem = np.array(healthy_memory + tbi_memory)

# Group comparison: t-test for each feature
print("\n--- Group Comparison: Healthy vs TBI ---")
sig_features = []
for i, fname in enumerate(feature_names):
    t, p = stats.ttest_ind(X_healthy[:,i], X_tbi[:,i])
    if p < 0.05:
        sig_features.append((fname, t, p))

sig_features.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"Significant features (p<0.05): {len(sig_features)}/{len(feature_names)}")
for fname, t, p in sig_features[:5]:
    direction = "↑healthy" if t > 0 else "↑TBI"
    print(f"  {fname}: t={t:.2f}, p={p:.4f} {direction}")

# Save results for plotting
# to this:

np.save('healthy_filtrations_euler.npy',
        np.array([f['euler'] for f in healthy_filtrations]))
np.save('tbi_filtrations_euler.npy',
        np.array([f['euler'] for f in tbi_filtrations]))
np.save('X_all.npy', X_all)
np.save('y_all.npy', y_all)
np.save('cog_iq.npy', cog_iq)
np.save('cog_mem.npy', cog_mem)
np.save('epsilon_steps.npy', epsilon_steps)

# Save feature names
with open('feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))

print("\nData saved. Running classification...")

# Logistic regression classifier
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
clf = LogisticRegression(max_iter=1000)
cv_scores = cross_val_score(clf, X_scaled, y_all, cv=5, scoring='roc_auc')
print(f"Classification AUC (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Correlation with IQ
print("\n--- Correlation with IQ scores ---")
for fname in [s[0] for s in sig_features[:5]]:
    idx = feature_names.index(fname)
    r, p = stats.pearsonr(X_all[:,idx], cog_iq)
    print(f"  {fname}: r={r:.3f}, p={p:.4f}")

print("\nAll done!")


--- Group Comparison: Healthy vs TBI ---
Significant features (p<0.05): 14/15
  euler_mean: t=292.97, p=0.0000 ↑healthy
  euler_profile_norm: t=263.82, p=0.0000 ↑healthy
  euler_std: t=227.37, p=0.0000 ↑healthy
  betti1_mean: t=185.21, p=0.0000 ↑healthy
  euler_range: t=136.14, p=0.0000 ↑healthy

Data saved. Running classification...
Classification AUC (5-fold CV): 1.000 ± 0.000

--- Correlation with IQ scores ---
  euler_mean: r=0.533, p=0.0000
  euler_profile_norm: r=0.534, p=0.0000
  euler_std: r=0.535, p=0.0000
  betti1_mean: r=0.535, p=0.0000
  euler_range: r=0.535, p=0.0000

All done!


## 5. Visualization
*9-panel figure showing Euler profiles, group differences, ROC curve,
Euler similarity matrix, and cognitive score correlations*

In [10]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Load saved data
healthy_euler = np.load('healthy_filtrations_euler.npy')
tbi_euler = np.load('tbi_filtrations_euler.npy')
X_all = np.load('X_all.npy')
y_all = np.load('y_all.npy')
cog_iq = np.load('cog_iq.npy')
cog_mem = np.load('cog_mem.npy')
epsilon_steps = np.load('epsilon_steps.npy')
with open('feature_names.txt') as f:
    feature_names = f.read().splitlines()

def euler_entropy(euler):
    with np.errstate(divide='ignore'):
        return np.where(euler != 0, np.log(np.abs(euler.astype(float))), np.nan)

# ============================================================
# FIGURE 1: Full analysis dashboard
# ============================================================
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#0f1117')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

BLUE = '#4C9BE8'
RED  = '#E85C5C'
GOLD = '#F0C040'
GREEN= '#5CB85C'
GRAY = '#888888'
BG   = '#1a1d27'
TEXT = '#e0e0e0'

def style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_color('#444')
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    if title: ax.set_title(title, color=TEXT, fontsize=9, fontweight='bold', pad=6)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8)

# ---- Plot 1: Euler characteristic profiles ----
ax1 = fig.add_subplot(gs[0, 0])
for i in range(min(20, len(healthy_euler))):
    ax1.plot(epsilon_steps, healthy_euler[i], color=BLUE, alpha=0.15, lw=0.8)
for i in range(min(20, len(tbi_euler))):
    ax1.plot(epsilon_steps, tbi_euler[i], color=RED, alpha=0.15, lw=0.8)
ax1.plot(epsilon_steps, np.mean(healthy_euler, axis=0), color=BLUE, lw=2, label='Healthy mean')
ax1.plot(epsilon_steps, np.mean(tbi_euler, axis=0), color=RED, lw=2, linestyle='--', label='TBI mean')
ax1.axhline(0, color=GOLD, lw=0.8, linestyle=':')
ax1.legend(fontsize=7, facecolor=BG, labelcolor=TEXT)
style_ax(ax1, 'Euler Characteristic χ(ε)', 'ε threshold', 'χ')

# ---- Plot 2: Euler entropy profiles ----
ax2 = fig.add_subplot(gs[0, 1])
healthy_entropy_mean = np.nanmean([euler_entropy(h) for h in healthy_euler], axis=0)
tbi_entropy_mean = np.nanmean([euler_entropy(t) for t in tbi_euler], axis=0)
for i in range(min(15, len(healthy_euler))):
    ax2.plot(epsilon_steps, euler_entropy(healthy_euler[i]), color=BLUE, alpha=0.12, lw=0.8)
for i in range(min(15, len(tbi_euler))):
    ax2.plot(epsilon_steps, euler_entropy(tbi_euler[i]), color=RED, alpha=0.12, lw=0.8)
ax2.plot(epsilon_steps, healthy_entropy_mean, color=BLUE, lw=2, label='Healthy')
ax2.plot(epsilon_steps, tbi_entropy_mean, color=RED, lw=2, linestyle='--', label='TBI')
ax2.legend(fontsize=7, facecolor=BG, labelcolor=TEXT)
style_ax(ax2, 'Euler Entropy Sχ = ln|χ|', 'ε threshold', 'Sχ')

# ---- Plot 3: Feature distributions (violin) ----
ax3 = fig.add_subplot(gs[0, 2])
feat_idx = feature_names.index('betti1_mean')
healthy_vals = X_all[y_all==0, feat_idx]
tbi_vals = X_all[y_all==1, feat_idx]
vp = ax3.violinplot([healthy_vals, tbi_vals], positions=[1,2], showmedians=True)
for body, color in zip(vp['bodies'], [BLUE, RED]):
    body.set_facecolor(color); body.set_alpha(0.6)
vp['cmedians'].set_color(GOLD)
vp['cbars'].set_color(TEXT); vp['cmaxes'].set_color(TEXT); vp['cmins'].set_color(TEXT)
ax3.set_xticks([1,2]); ax3.set_xticklabels(['Healthy','TBI'], color=TEXT, fontsize=8)
t, p = stats.ttest_ind(healthy_vals, tbi_vals)
ax3.set_title(f'β₁ mean: p={p:.2e}', color=TEXT, fontsize=9, fontweight='bold', pad=6)
style_ax(ax3, ylabel='β₁ mean value')

# ---- Plot 4: Feature importance (t-statistics) ----
ax4 = fig.add_subplot(gs[1, 0])
t_stats = []
for i in range(len(feature_names)):
    t, p = stats.ttest_ind(X_all[y_all==0, i], X_all[y_all==1, i])
    t_stats.append(abs(t))
sorted_idx = np.argsort(t_stats)
colors_bar = [BLUE if t_stats[i] > 0 else RED for i in sorted_idx]
bars = ax4.barh(range(len(feature_names)), [t_stats[i] for i in sorted_idx],
                color=[BLUE if t_stats[i] > 50 else GREEN if t_stats[i] > 10 else GRAY
                       for i in sorted_idx], alpha=0.8)
ax4.set_yticks(range(len(feature_names)))
ax4.set_yticklabels([feature_names[i][:18] for i in sorted_idx], fontsize=6.5)
ax4.axvline(2, color=GOLD, lw=0.8, linestyle=':')
style_ax(ax4, 'Feature Discrimination Power', '|t-statistic|', 'Feature')

# ---- Plot 5: ROC curve ----
ax5 = fig.add_subplot(gs[1, 1])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_scaled, y_all)
y_prob = clf.predict_proba(X_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_all, y_prob)
auc = roc_auc_score(y_all, y_prob)
ax5.plot(fpr, tpr, color=BLUE, lw=2, label=f'AUC = {auc:.3f}')
ax5.plot([0,1],[0,1], color=GRAY, linestyle='--', lw=1)
ax5.fill_between(fpr, tpr, alpha=0.15, color=BLUE)
ax5.legend(fontsize=8, facecolor=BG, labelcolor=TEXT)
style_ax(ax5, 'ROC: TBI vs Healthy Classification', 'False Positive Rate', 'True Positive Rate')

# ---- Plot 6: Euler similarity heatmap ----
ax6 = fig.add_subplot(gs[1, 2])
# Euler similarity = cosine similarity between Euler profiles
euler_profiles = np.vstack([healthy_euler[:20], tbi_euler[:20]])
norms = np.linalg.norm(euler_profiles, axis=1, keepdims=True)
normalized = euler_profiles / (norms + 1e-10)
similarity_matrix = normalized @ normalized.T
im = ax6.imshow(similarity_matrix, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
ax6.axhline(19.5, color=GOLD, lw=1.5)
ax6.axvline(19.5, color=GOLD, lw=1.5)
ax6.set_xticks([10, 30]); ax6.set_xticklabels(['Healthy\n(1-20)', 'TBI\n(21-40)'],
                                                 color=TEXT, fontsize=7)
ax6.set_yticks([10, 30]); ax6.set_yticklabels(['Healthy', 'TBI'], color=TEXT, fontsize=7)
plt.colorbar(im, ax=ax6, shrink=0.8).ax.tick_params(colors=TEXT, labelsize=7)
style_ax(ax6, 'Euler Similarity Matrix', '', '')

# ---- Plot 7: Correlation entropy AUC vs IQ ----
ax7 = fig.add_subplot(gs[2, 0])
feat_idx2 = feature_names.index('entropy_auc')
entropy_auc_vals = X_all[:, feat_idx2]
ax7.scatter(entropy_auc_vals[y_all==0], cog_iq[y_all==0],
            color=BLUE, alpha=0.6, s=30, label='Healthy')
ax7.scatter(entropy_auc_vals[y_all==1], cog_iq[y_all==1],
            color=RED, alpha=0.6, s=30, marker='^', label='TBI')
# Regression line
m, b, r, p, _ = stats.linregress(entropy_auc_vals, cog_iq)
x_line = np.linspace(entropy_auc_vals.min(), entropy_auc_vals.max(), 100)
ax7.plot(x_line, m*x_line + b, color=GOLD, lw=1.5, label=f'r={r:.2f}, p={p:.3f}')
ax7.legend(fontsize=7, facecolor=BG, labelcolor=TEXT)
style_ax(ax7, 'Euler Entropy AUC vs IQ', 'Entropy AUC', 'IQ Score')

# ---- Plot 8: Betti-1 max vs Memory score ----
ax8 = fig.add_subplot(gs[2, 1])
feat_idx3 = feature_names.index('betti1_max')
betti1_max_vals = X_all[:, feat_idx3]
ax8.scatter(betti1_max_vals[y_all==0], cog_mem[y_all==0],
            color=BLUE, alpha=0.6, s=30, label='Healthy')
ax8.scatter(betti1_max_vals[y_all==1], cog_mem[y_all==1],
            color=RED, alpha=0.6, s=30, marker='^', label='TBI')
m2, b2, r2, p2, _ = stats.linregress(betti1_max_vals, cog_mem)
x_line2 = np.linspace(betti1_max_vals.min(), betti1_max_vals.max(), 100)
ax8.plot(x_line2, m2*x_line2 + b2, color=GOLD, lw=1.5, label=f'r={r2:.2f}, p={p2:.3f}')
ax8.legend(fontsize=7, facecolor=BG, labelcolor=TEXT)
style_ax(ax8, 'β₁ max vs Memory Score', 'β₁ max', 'Memory Score')

# ---- Plot 9: Phase transition location comparison ----
ax9 = fig.add_subplot(gs[2, 2])
feat_idx4 = feature_names.index('first_transition')
healthy_trans = X_all[y_all==0, feat_idx4]
tbi_trans = X_all[y_all==1, feat_idx4]
ax9.hist(healthy_trans, bins=15, color=BLUE, alpha=0.6, label='Healthy', density=True)
ax9.hist(tbi_trans, bins=15, color=RED, alpha=0.6, label='TBI', density=True)
ax9.axvline(np.mean(healthy_trans), color=BLUE, lw=2, linestyle='--')
ax9.axvline(np.mean(tbi_trans), color=RED, lw=2, linestyle='--')
t_tr, p_tr = stats.ttest_ind(healthy_trans, tbi_trans)
ax9.legend(fontsize=7, facecolor=BG, labelcolor=TEXT)
style_ax(ax9, f'Phase Transition εc: p={p_tr:.3f}', 'εc location', 'Density')

# Main title
fig.suptitle('Synthetic TBI Brain Network — Topological Analysis Pipeline',
             color=TEXT, fontsize=13, fontweight='bold', y=0.98)

plt.savefig('tbi_tda_analysis.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print("Figure saved!")

Figure saved!
